# Detalles de TTree

# Estructura y terminología de los archivos ROOT

Un archivo ROOT ([ROOT TFile](https://root.cern.ch/doc/master/classTFile.html), [uproot.ReadOnlyFile](https://uproot.readthedocs.io/en/latest/uproot.reading.ReadOnlyFile.html)) es como un pequeño sistema de archivos que contiene directorios anidados ([ROOT TDirectory](https://root.cern.ch/doc/master/classTDirectory.html), [uproot.ReadOnlyDirectory](https://uproot.readthedocs.io/en/latest/uproot.reading.ReadOnlyDirectory.html)). En Uproot, los directorios anidados se presentan como diccionarios anidados.

Cualquier instancia de clase ([ROOT TObject](https://root.cern.ch/doc/master/classTObject.html), [uproot.Model](https://uproot.readthedocs.io/en/latest/uproot.model.Model.html)) puede almacenarse en un directorio, incluidos tipos como histogramas (por ejemplo, [ROOT TH1](https://root.cern.ch/doc/master/classTH1.html), [uproot.behaviors.TH1.TH1](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TH1.TH1.html)).

Una de estas clases, TTree ([ROOT TTree](https://root.cern.ch/doc/master/classTTree.html), [uproot.TTree](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html)), es una puerta de entrada a conjuntos de datos grandes. Un TTree es algo parecido a un DataFrame de Pandas en el sentido de que representa una tabla de datos. Las columnas se llaman TBranches ([ROOT TBranch](https://root.cern.ch/doc/master/classTBranch.html), [uproot.TBranch](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TBranch.TBranch.html)), que pueden estar anidadas (a diferencia de Pandas), y los datos pueden tener cualquier tipo de C++ (a diferencia de Pandas, que puede almacenar cualquier tipo de Python).

Un TTree a menudo es demasiado grande para caber en la memoria, y a veces (raramente) incluso un solo TBranch es demasiado grande para caber en la memoria. Por eso cada TBranch se divide en TBaskets ([ROOT TBasket](https://root.cern/doc/master/classTBasket.html), [uproot.models.TBasket.Model_TBasket](https://uproot.readthedocs.io/en/latest/uproot.models.TBasket.Model_TBasket.html)), que son "lotes" de datos. (Estos son los mismos lotes que escribe cada llamada a `extend` en la lección anterior). Los TBaskets son la unidad más pequeña que se puede leer de un TTree: si deseas leer la primera entrada, tienes que leer el primer TBasket.

![terminología](img/terminology.png)

Como analista de datos, probablemente te ocuparás de los TTrees y TBranches de manera directa, pero solo de los TBaskets cuando surjan problemas de eficiencia. Los archivos con TBaskets grandes pueden requerir mucha memoria para leerse; los archivos con TBaskets pequeños serán más lentos de leer (en ROOT también, pero especialmente en Uproot). Los TBaskets del orden de megabytes suelen ser ideales.

# Ejemplos con un TTree grande

[Este archivo](http://opendata.web.cern.ch/record/12341) tiene 2.1 GB y está alojado en el Portal de Datos Abiertos del CERN.

In [ ]:
import uproot

url_archivo = "root://eospublic.cern.ch//eos/opendata/cms/derived-data/AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root"

# Si estás en Windows o no tienes XRootD instalado, puedes usar esta url en su lugar
# url_archivo = "https://root.cern/files/rootbench/Run2012BC_DoubleMuParked_Muons.root"

archivo = uproot.open(url_archivo)
archivo.classnames()

:::{note} ¿Por qué el `;74` y el `;75`?
Tal vez te hayas preguntado por los números que van después de los puntos y comas. Estos son los "números de ciclo" de ROOT, que permiten distinguir objetos con el mismo nombre. Se utilizan cuando un objeto necesita sobrescribirse a medida que crece sin perder la última copia válida de ese objeto, de modo que un archivo ROOT pueda leerse incluso si el proceso de escritura falló a mitad de camino.

En este caso, la última versión de este TTree es el número 75, y el número 74 es la penúltima.

Si no especificas números de ciclo, Uproot seleccionará el último por ti, que es casi siempre lo que quieres. (En otras palabras, puedes ignorarlos).
:::

Simplemente pedir el objeto [uproot.TTree](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html) e imprimirlo *no* lee todo el conjunto de datos.

In [ ]:
tree = archivo["Events"]
tree.show()

## Leer una parte de un TTree

En la lección anterior aprendimos que la forma más directa de leer un TBranch es llamando a [uproot.TBranch.array](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TBranch.TBranch.html#array).

In [ ]:
# sin entry_stop, tomará mucho tiempo e incluso podría fallar
tree["nMuon"].array(entry_stop=10_000)

Leer el TBranch completo tomaría mucho tiempo, porque habría que enviar todos sus datos a través de la red. Por eso la celda anterior pasa `entry_stop`.

De forma más general, establece `entry_start` y `entry_stop` en el rango que desees. `entry_start` es inclusivo, `entry_stop` es exclusivo, y la primera entrada se indexa con `0`, al igual que las rebanadas en una interfaz de array (primera lección). Uproot solo lee tantos TBaskets como sean necesarios para proporcionar estas entradas.

In [ ]:
tree["nMuon"].array(entry_start=1_000, entry_stop=2_000)

Estos son los bloques de construcción de un lector de datos en paralelo: cada uno es responsable de una rebanada diferente. (Consulta también [uproot.TTree.num_entries_for](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#num-entries-for) y [uproot.TTree.common_entry_offsets](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#common-entry-offsets), que se pueden usar para elegir `entry_start`/`entry_stop` de manera óptima).

## Leer múltiples TBranches a la vez

Supongamos que sabes que vas a necesitar todos los TBranches de muones. Pedirlos en una sola solicitud es más eficiente que pedir cada TBranch individualmente, porque el servidor puede ir leyendo del disco los TBaskets posteriores mientras los TBaskets anteriores se te envían a través de la red. Mientras que un TBranch tiene un método `array`, el TTree tiene un método `arrays` (en plural) para obtener múltiples arrays.

In [ ]:
muones = tree.arrays(
    ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"], entry_stop=1_000
)
muones

Ahora los cinco TBranches están en la salida, `muones`, que es un Awkward Array. Un Awkward Array de múltiples TBranches tiene una interfaz similar a un diccionario, por lo que podemos obtener cada variable de él así

In [ ]:
muones["Muon_pt"]

In [ ]:
muones["Muon_eta"]

In [ ]:
muones["Muon_phi"]  # etc.

::::{caution} ¡Cuidado! ¡Es `tree.arrays` lo que realmente lee los datos!
Si no tienes cuidado con la llamada a [uproot.TTree.arrays](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#arrays), podrías terminar esperando mucho tiempo por datos que no necesitas, o podrías quedarte sin memoria. Leer todo con

```python
todo = tree.arrays()
```

y luego seleccionar los arrays que deseas generalmente no es una buena idea. Como mínimo, establece un `entry_stop`.
::::

## Seleccionar TBranches por nombre

Supongamos que tienes muchos TBranches de muones y no quieres enumerarlos todos. Tanto [uproot.TTree.keys](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#keys) como [uproot.TTree.arrays](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#arrays) aceptan un argumento `filter_name` que puede seleccionarlos de varias maneras (consulta la documentación). En particular, es recomendable usar primero `keys` para saber qué TBranches coinciden con tu filtro, seguido de `arrays` para leerlas realmente.

In [ ]:
tree.keys(filter_name="Muon_*")

In [ ]:
tree.arrays(filter_name="Muon_*", entry_stop=1_000)

(También hay `filter_typename` y `filter_branch` para más opciones).

## Escalar el análisis y hacer un gráfico

La mejor manera de entender lo que estás haciendo es experimentar con conjuntos de datos pequeños y luego escalarlos. Aquí tomamos 1000 eventos y calculamos las masas de los dimuones.

In [ ]:
muones = tree.arrays(entry_stop=1_000)
corte = muones["nMuon"] == 2

pt0 = muones["Muon_pt", corte, 0]
pt1 = muones["Muon_pt", corte, 1]
eta0 = muones["Muon_eta", corte, 0]
eta1 = muones["Muon_eta", corte, 1]
phi0 = muones["Muon_phi", corte, 0]
phi1 = muones["Muon_phi", corte, 1]

import numpy as np

masa = np.sqrt(2 * pt0 * pt1 * (np.cosh(eta0 - eta1) - np.cos(phi0 - phi1)))

import hist

histmasa = hist.Hist(hist.axis.Regular(120, 0, 120, label="masa [GeV]"))
histmasa.fill(masa)
histmasa.plot();

Eso funcionó (hay un pico del Z). Ahora, para hacer esto sobre todo el archivo, debemos tener más cuidado con lo que estamos leyendo,

In [ ]:
tree.keys(filter_name=["nMuon", "/Muon_(pt|eta|phi)/"])

y acumular datos gradualmente con [uproot.TTree.iterate](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html#iterate). Esto maneja `entry_start`/`entry_stop` en un bucle.

In [ ]:
histmasa = hist.Hist(hist.axis.Regular(120, 0, 120, label="masa [GeV]"))

# Pon entry_stop igual a tree.num_entries para procesar todo el archivo, pero toma mucho tiempo
entry_stop = 250_000
total = min(entry_stop, tree.num_entries)

procesadas = 0
for muones in tree.iterate(filter_name=["nMuon", "/Muon_(pt|eta|phi)/"], step_size=50_000, entry_stop=entry_stop):
    corte = muones["nMuon"] == 2
    pt0 = muones["Muon_pt", corte, 0]
    pt1 = muones["Muon_pt", corte, 1]
    eta0 = muones["Muon_eta", corte, 0]
    eta1 = muones["Muon_eta", corte, 1]
    phi0 = muones["Muon_phi", corte, 0]
    phi1 = muones["Muon_phi", corte, 1]
    masa = np.sqrt(2 * pt0 * pt1 * (np.cosh(eta0 - eta1) - np.cos(phi0 - phi1)))
    histmasa.fill(masa)
    procesadas += len(muones)
    print(f"procesadas {procesadas} de {total} entradas")

histmasa.plot();

## Pasar los datos a NumPy o Pandas

En todos los ejemplos anteriores, los métodos `array`, `arrays` e `iterate` devuelven Awkward Arrays. La librería Awkward Array es útil exactamente para este tipo de datos (arrays irregulares: más sobre esto en la próxima lección), pero es posible que estés trabajando con librerías que solo reconocen arrays de NumPy o DataFrames de Pandas.

Utiliza `library="np"` o `library="pd"` para obtener NumPy o Pandas, respectivamente.

In [ ]:
tree["nMuon"].array(library="np", entry_stop=10_000)

In [ ]:
tree.arrays(library="np", entry_stop=10_000)

In [ ]:
tree.arrays(library="pd", entry_stop=10_000)

NumPy es excelente para datos no irregulares como el TBranch `"nMuon"`, pero tiene que representar un número desconocido de muones por evento como un array de arrays de NumPy (es decir, objetos de Python).

Se puede hacer que Pandas represente múltiples partículas por evento colocando esta estructura en un [pd.MultiIndex](https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html), pero no cuando el DataFrame contiene más de un tipo de partícula (por ejemplo, muones *y* electrones). Usa DataFrames separados para estos casos. Si ayuda, ten en cuenta que hay otra ruta hacia los DataFrames: leer los datos como un Awkward Array y llamar a [ak.to_dataframe](https://awkward-array.org/doc/main/reference/generated/ak.to_dataframe.html) sobre él. (Algunos métodos usan más memoria que otros; Pandas tiende a consumir una cantidad de memoria inusualmente alta).

O usa Awkward Arrays (próxima lección).